# 02 · Silver: cleaned and classified ads

Reads the raw JSON files in `lh_bronze`, flattens them into columns, deduplicates, classifies each ad into a data role, and writes the result to `lh_silver.dbo.ads`.

- **Input:** `lh_bronze` → `Files/bronze/historical/term=…/year=…/month=…/ads.json` (896 files, 16,458 ad rows)
- **Output:** `lh_silver.dbo.ads` — one row per ad, 11,718 rows
- **How to run:** *Run all*. The table is overwritten, so re-running gives the same result.
- **Lakehouses:** `lh_bronze` must be the default (the notebook reads from a relative path); `lh_silver` must also be attached.

## Design notes

- **Search terms decide what was fetched, not what an ad is.** The API expands a term into its own concepts: *BI developer* became the skill "bi" plus the occupation "systems developer", which pulled in full-stack developers and even a laboratory engineer. The role is therefore decided here, from the ad headline.
- **Occupation codes are not reliable either.** Ads titled "Data Analyst" carry occupation labels such as *Database developer*, *Systems analyst* and *Public administration officer*, and BI developer has no occupation code at all.
- **Headlines are free text**, so matching is done on a normalised headline (lower case, punctuation replaced by spaces) with word-boundary patterns: `BI- utvecklare`, `BI/DW Utvecklare` and `BI MICROSOFT DEVELOPER` all have to match, while `bibliotek` and `Data Centre Engineering` must not.
- **Rules are ordered from specific to general.** *Analytics Engineer* is tested before *Data Engineer*, otherwise it would never match.
- **Ads that match no rule are dropped** (1,662 of 13,380). They are mostly from the two noisy search terms. Bronze keeps everything, so the scope can be changed by re-running this notebook.
- **Overlap between search terms is kept as information.** 18.7% of ad rows were found via more than one term; the set of terms is stored in `search_terms`, since e.g. an ad found by both *data engineer* and *data scientist* says something about the role.
- **Known limitation:** the workplace region is the one stated in the ad. Staffing agencies sometimes give their own office, e.g. an ad headlined "BI Developer till Telenor, Karlskrona" (Blekinge) is registered in Kronobergs län.
- **Accuracy:** in a manual sample of 30 classified ads, 1–2 looked wrong (a teaching post on a Data Scientist programme; an ad headlined "Datascientist Machine learning Engineer", which is genuinely ambiguous). That is roughly 95% precision on a small sample.


## 1. Setup

In [1]:
from pyspark.sql import functions as F

BRONZE_PATH = "Files/bronze/historical"   # relative to the default lakehouse (lh_bronze)
SILVER_TABLE = "lh_silver.dbo.ads"

StatementMeta(, b3ec3586-b838-4f23-b691-077073e16573, 3, Finished, Available, Finished, False)

## 2. Read bronze

Spark reads the `term=…/year=…/month=…` folder names as columns, which is why bronze is partitioned that way. `multiLine` is needed because each file is one JSON object, not one object per line.

In [2]:
raw = spark.read.option("multiLine", "true").json(BRONZE_PATH)
print("Files read:", raw.count())

# One row per ad: explode the list of hits inside each file.
ads = raw.select(
    "term", "year", "month",
    F.col("_meta.search_term").alias("search_term"),
    F.explode("hits").alias("ad"),
)

n_rows = ads.count()
n_unique = ads.select("ad.id").distinct().count()
print(f"Ad rows: {n_rows}")
print(f"Unique ad ids: {n_unique}")
print(f"Overlap between search terms: {n_rows - n_unique} ({(n_rows - n_unique) / n_rows:.1%})")

StatementMeta(, b3ec3586-b838-4f23-b691-077073e16573, 4, Finished, Available, Finished, False)

Files read: 896
Ad rows: 16458
Unique ad ids: 13380
Overlap between search terms: 3078 (18.7%)


## 3. Which search terms found each ad

Collected before deduplication, because the combination of terms is itself a signal about the role.

In [3]:
terms_per_ad = (ads.groupBy("ad.id")
    .agg(F.sort_array(F.collect_set("search_term")).alias("search_terms"))
    .withColumn("n_terms", F.size("search_terms"))
    .withColumnRenamed("id", "ad_id"))

terms_per_ad.groupBy("n_terms").count().orderBy("n_terms").show()

StatementMeta(, b3ec3586-b838-4f23-b691-077073e16573, 5, Finished, Available, Finished, False)

+-------+-----+
|n_terms|count|
+-------+-----+
|      1|12276|
|      2| 1061|
|      3|   43|
+-------+-----+



## 4. Flatten to columns

Pick the fields the analysis needs out of the nested structure, give them readable names, parse dates, and keep one row per ad.

In [4]:
flat = (ads
    .select(
        F.col("ad.id").alias("ad_id"),
        F.col("ad.headline").alias("headline"),
        F.col("ad.description.text").alias("description"),
        F.col("ad.employer.name").alias("employer_name"),
        F.col("ad.employer.organization_number").alias("employer_orgnr"),
        F.col("ad.workplace_address.region").alias("region"),
        F.col("ad.workplace_address.municipality").alias("municipality"),
        F.col("ad.occupation.label").alias("occupation"),
        F.col("ad.occupation_group.label").alias("occupation_group"),
        F.col("ad.occupation_field.label").alias("occupation_field"),
        F.col("ad.employment_type.label").alias("employment_type"),
        F.col("ad.working_hours_type.label").alias("working_hours_type"),
        F.col("ad.number_of_vacancies").alias("n_vacancies"),
        F.to_timestamp("ad.publication_date").alias("published_at"),
        F.to_timestamp("ad.last_publication_date").alias("last_published_at"),
        F.to_timestamp("ad.application_deadline").alias("application_deadline"),
        F.col("ad.removed").alias("removed"),
        F.col("ad.webpage_url").alias("webpage_url"),
    )
    # Safe because bronze is unmodified: the duplicate rows are identical ads
    # found through different search terms.
    .dropDuplicates(["ad_id"])
    .join(terms_per_ad, "ad_id", "left"))

print("Rows after dedup:", flat.count())

StatementMeta(, b3ec3586-b838-4f23-b691-077073e16573, 6, Finished, Available, Finished, False)

Rows after dedup: 13380


## 5. Classify into roles

Rules are applied top-down, most specific first, on a normalised headline.

In [5]:
# "BI- utvecklare" -> "bi utvecklare": lower case, any punctuation becomes a space.
norm = F.regexp_replace(F.lower(F.col("headline")), r"[^a-z0-9åäö]+", " ")

classified = (flat
    .withColumn("headline_norm", norm)
    .withColumn("role",
        F.when(F.col("headline_norm").rlike(r"analytics engineer"), "Analytics Engineer")
         .when(F.col("headline_norm").rlike(r"\b(data|big data|ml|machine learning)\b.*\bengineer\b|dataingenjör|datatekniker"), "Data Engineer")
         .when(F.col("headline_norm").rlike(r"data scientist|datavetare|machine learning"), "Data Scientist")
         .when(F.col("headline_norm").rlike(r"data analyst|dataanalytiker|dataanalys|\bdata analys"), "Data Analyst")
         .when(F.col("headline_norm").rlike(r"\bbi\b|\bdw\b|business intelligence|power bi|qlik|data warehouse|datalager"), "BI Developer")
         .otherwise(F.lit(None))))

classified.groupBy("role").count().orderBy(F.desc("count")).show(truncate=False)

StatementMeta(, b3ec3586-b838-4f23-b691-077073e16573, 7, Finished, Available, Finished, False)

+------------------+-----+
|role              |count|
+------------------+-----+
|Data Engineer     |4438 |
|Data Analyst      |2495 |
|BI Developer      |2361 |
|Data Scientist    |2201 |
|NULL              |1662 |
|Analytics Engineer|223  |
+------------------+-----+



## 6. Inspect the classification

Check both directions: what the rules miss, and whether what they catch is right. The second sample is meant to be read by a human — that is where the precision estimate in the design notes comes from.

In [6]:
print("Unclassified:", classified.filter("role is null").count())
classified.filter("role is null").select("headline", "search_terms").show(20, truncate=60)

classified.filter("role is not null").select("role", "headline").sample(0.02, seed=42).show(30, truncate=60)

StatementMeta(, b3ec3586-b838-4f23-b691-077073e16573, 8, Finished, Available, Finished, False)

Unclassified: 1662
+------------------------------------------------------------+-----------------------------+
|                                                    headline|                 search_terms|
+------------------------------------------------------------+-----------------------------+
|Kvantitativ analytiker för analys av hälsa och läkemedels...|             [data scientist]|
| Laboratorieingenjör till Octapharma, Kungsholmen, Stockholm|       ["analytics engineer"]|
|                                          Fullstack engineer|[BI developer, data engineer]|
|                          Utvecklare till Beijer Electronics|               [BI developer]|
|                               Digital Product Manager, Data|             [dataanalytiker]|
|    Systemutvecklare till Swedemount Sportswear & Fashion AB|               [BI developer]|
|                 Full stack developer with focus on frontend|               [BI developer]|
|         Applikationsutvecklare till vår kund på K

## 7. Write the silver table

In [7]:
silver = (classified
    .filter("role is not null")
    .drop("headline_norm")
    .withColumn("ingested_at", F.current_timestamp()))

(silver.write
    .mode("overwrite")                      # idempotent: re-running replaces the table
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE))

print("Rows written:", spark.table(SILVER_TABLE).count())

StatementMeta(, b3ec3586-b838-4f23-b691-077073e16573, 9, Finished, Available, Finished, False)

Rows written: 11718


## 8. Verify

In [8]:
t = spark.table(SILVER_TABLE)

print("Ads per role and year (2026 is partial):")
(t.groupBy(F.year("published_at").alias("year"), "role").count()
  .groupBy("year").pivot("role").sum("count").orderBy("year").show())

n_rows = t.count()
n_ids = t.select("ad_id").distinct().count()
n_no_role = t.filter("role is null").count()
n_no_date = t.filter("published_at is null").count()
n_no_text = t.filter("description is null or length(description) = 0").count()
print(f"Rows: {n_rows}, unique ids: {n_ids}, missing role: {n_no_role}, "
      f"missing publication date: {n_no_date}, missing description: {n_no_text}")

assert n_rows == n_ids, "Duplicate ad ids in the silver table"
assert n_no_role == 0, "Unclassified ads were written"
assert n_no_date == 0, "Ads without a publication date"

StatementMeta(, b3ec3586-b838-4f23-b691-077073e16573, 10, Finished, Available, Finished, False)

Ads per role and year (2026 is partial):
+----+------------------+------------+------------+-------------+--------------+
|year|Analytics Engineer|BI Developer|Data Analyst|Data Engineer|Data Scientist|
+----+------------------+------------+------------+-------------+--------------+
|2016|                 5|         123|          37|           30|            60|
|2017|                 7|         157|          63|           38|            86|
|2018|              NULL|         189|          99|          117|           151|
|2019|                 2|         270|         147|          207|           205|
|2020|                 4|         187|         139|          242|           198|
|2021|                27|         252|         305|          548|           319|
|2022|                23|         303|         371|          677|           297|
|2023|                40|         253|         422|          647|           253|
|2024|                31|         267|         406|          645|   